# SafeWalk Navigation — 8-Class YOLO Training Pipeline v7

## v7 전략
- 모델: yolov8n → **yolov8x** (최대 모델, 정확도 극대화)
- **2단계 학습**: Phase 1 (클래스별 개별 학습) → Phase 2 (통합 파인튜닝)
- **실시간 처리 최적화**: TensorRT export, half precision, 추론 속도 측정
- imgsz: 640 (실시간 처리 속도 유지)
- 클래스별 augmentation 강도 차별화

## 성능 목표
| Class | v6 AP@0.5 | v7 목표 |
|---|---|---|
| bicycle   | 48.0% ❌ | 70%+ |
| kickboard | 44.5% ❌ | 70%+ |
| bollard   | 73.7% ⚠️ | 85%+ |
| car       | 85.8% ✅ | 90%+ |
| motorcycle| 65.6% ⚠️ | 80%+ |
| stairs    | 70.8% ⚠️ | 85%+ |
| crosswalk | 73.6% ⚠️ | 85%+ |
| people    | 32.9% ❌ | 70%+ |

In [1]:
# ============================================================
# Cell 0: 패키지 설치 & 환경 설정
# ============================================================
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install",
                "ultralytics>=8.3.0", "roboflow",
                "opencv-python-headless", "matplotlib",
                "pandas", "pyyaml", "seaborn",
                "--quiet", "--upgrade"], check=True)

# nvrtc 심링크 자동 수정
nvrtc_src = "/usr/local/lib/python3.10/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0"
nvrtc_dst = "/usr/local/lib/libnvrtc-builtins.so.13.0"
if os.path.exists(nvrtc_src) and not os.path.exists(nvrtc_dst):
    subprocess.run(["ln", "-sf", nvrtc_src, nvrtc_dst], check=True)
    subprocess.run(["ldconfig"], check=True)
    print("nvrtc 심링크 설정 완료")

import shutil, random, yaml, time
from pathlib import Path
from collections import defaultdict

import cv2, numpy as np, torch
from ultralytics import YOLO

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM  = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE   = "0"
    print(f"GPU: {GPU_NAME} ({GPU_MEM:.1f} GB)")
else:
    DEVICE = "cpu"
    print("CPU 모드")

PROJECT_ROOT  = Path("/root/pbl2-main")
DATA_DIR      = PROJECT_ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
MERGED_DIR    = DATA_DIR / "merged"
PROC_DIR      = DATA_DIR / "processed"
CONFIG_DIR    = PROJECT_ROOT / "configs"
MODELS_DIR    = PROJECT_ROOT / "models"
PHASE1_DIR    = MODELS_DIR / "phase1_per_class"  # 클래스별 모델 저장
PHASE2_DIR    = MODELS_DIR / "safewalk_v7"        # 최종 통합 모델

for d in [RAW_DIR, MERGED_DIR, PROC_DIR, CONFIG_DIR, MODELS_DIR, PHASE1_DIR, PHASE2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CONFIG_DIR / "dataset_8class.yaml"

CLASS_NAMES = [
    "bicycle",    # 0
    "kickboard",  # 1
    "bollard",    # 2
    "car",        # 3
    "motorcycle", # 4
    "stairs",     # 5
    "crosswalk",  # 6
    "people",     # 7
]
NUM_CLASSES = len(CLASS_NAMES)
RF_API_KEY  = os.environ.get("ROBOFLOW_API_KEY", "ecwQGp9pcTxUjqPNhjXG")

# ---- VRAM에 따른 배치 크기 (yolov8x는 메모리 많이 씀) ----
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 16 if vram_gb >= 24 else 8 if vram_gb >= 16 else 4 if vram_gb >= 10 else 2
    print(f"VRAM: {vram_gb:.1f} GB → batch={BATCH}")
else:
    BATCH = 2

print("="*60)
print(" SafeWalk v7 Setup Complete")
print(" 모델: YOLOv8x | 전략: 클래스별 학습 → 통합 파인튜닝")
print("="*60)

GPU: NVIDIA GeForce RTX 4090 (25.8 GB)
VRAM: 25.8 GB → batch=16
 SafeWalk v7 Setup Complete
 모델: YOLOv8x | 전략: 클래스별 학습 → 통합 파인튜닝


## Cell 1 — 데이터셋 다운로드

In [2]:
# ============================================================
# Cell 1: Roboflow 데이터셋 다운로드 (이미 있으면 스킵)
# ============================================================
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)

RF_DATASETS = [
    # bollard + stairs + crosswalk
    {
        "workspace": "esera", "project": "bollards-crosswalk-stairs", "version": 1,
        "class_map": {
            "bollard": 2, "bollards": 2, "traffic-cone": 2,
            "stairs": 5, "stair": 5, "step": 5, "steps": 5,
            "staircase": 5, "escalator": 5,
            "crosswalk": 6, "zebra-crossing": 6, "pedestrian-crossing": 6,
        },
        "out_name": "bollard_stairs_crosswalk",
    },
    # bicycle + kickboard + people
    {
        "workspace": "engs-89", "project": "bikes-pedestrians-scooters-wheelchairs", "version": 1,
        "class_map": {
            "scooter": 1, "e-scooter": 1, "electric-scooter": 1,
            "kickboard": 1, "kick-scooter": 1,
            "bicycle": 0, "bike": 0,
            "person": 7, "pedestrian": 7, "human": 7,
        },
        "out_name": "kickboard_bicycle_people",
    },
    # motorcycle x3
    {"workspace": "bollard",       "project": "motorcycle-l9wuv", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_1"},
    {"workspace": "aflakha",       "project": "motorcycle-illvg", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_2"},
    {"workspace": "sagemcom-j0etp","project": "motorcycle-m1pbt", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_3"},
    # stairs x2
    {"workspace": "stairs-n9kkx",          "project": "stairs-lusiz",    "version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5},
     "out_name": "stairs_1"},
    {"workspace": "escalatorstairsdetection","project": "escalator-stairs","version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5, "escalators": 5},
     "out_name": "stairs_2"},
    # crosswalk x2
    {"workspace": "wqwdas", "project": "crosswalk-1elwe", "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6,
                   "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_1"},
    {"workspace": "kmong", "project": "crosswalk-i59cr", "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6,
                   "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_2"},
    # kickboard 전용 x4
    {"workspace": "vkie39",        "project": "kickboard-j1mfx", "version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1,
                   "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_only_1"},
    {"workspace": "inha-univ-vgzgz","project": "kickboard-ibhkj", "version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1,
                   "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_only_2"},
    {"workspace": "school-zr91f",  "project": "kickboard-er5c4", "version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1,
                   "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_only_3"},
    {"workspace": "dklyolo1",      "project": "kickboard-dfkjo", "version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1,
                   "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_only_4"},
    # bollard 전용 x2
    {"workspace": "shin-7ty4z", "project": "bollard-ds7rv", "version": 1,
     "class_map": {"bollard": 2, "bollards": 2, "traffic-cone": 2, "cone": 2, "post": 2},
     "out_name": "bollard_only_1"},
    {"workspace": "facility",   "project": "bollard-3icj2",  "version": 1,
     "class_map": {"bollard": 2, "bollards": 2, "traffic-cone": 2, "cone": 2, "post": 2},
     "out_name": "bollard_only_2"},
    # pedestrian 전용 x3
    {"workspace": "project-fy",              "project": "pedestrian-detection-gpggm", "version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_only_1"},
    {"workspace": "samproject1",             "project": "pedestrian-detection-vsmzo", "version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_only_2"},
    {"workspace": "pedestriantracking-0t91d","project": "pedestrian-detection-fqlmr","version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_only_3"},
]


def parse_yaml_names(yaml_path):
    try:
        with open(yaml_path, encoding="utf-8") as f:
            dy = yaml.safe_load(f)
        raw = dy.get("names", [])
        if isinstance(raw, list):
            return {i: n for i, n in enumerate(raw)}
        elif isinstance(raw, dict):
            return {int(k): str(v) for k, v in raw.items()}
    except:
        pass
    return {}


def build_id_map(orig_names, class_map):
    id_map = {}
    for orig_id, orig_name in orig_names.items():
        for alias, our_id in class_map.items():
            if str(orig_name).lower().strip() == alias.lower().strip():
                id_map[int(orig_id)] = our_id
                break
    if not id_map and orig_names:
        for oid, nid in zip(sorted(orig_names), sorted(set(class_map.values()))):
            id_map[oid] = nid
    return id_map


def remap_and_copy(src_root, class_map, out_name):
    out_img = RAW_DIR / out_name / "images"
    out_lbl = RAW_DIR / out_name / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    yaml_path = src_root / "data.yaml"
    if not yaml_path.exists():
        found = list(src_root.rglob("data.yaml"))
        yaml_path = found[0] if found else None
    orig_names = parse_yaml_names(yaml_path) if yaml_path else {}
    print(f"     원본 클래스: {orig_names}")
    id_map = build_id_map(orig_names, class_map)
    print(f"     ID 매핑: {id_map}")
    if not id_map:
        print(f"     매핑 실패 — 스킵")
        return 0
    converted = 0
    for split in ["train", "valid", "test"]:
        lbl_dir = src_root / split / "labels"
        img_dir = src_root / split / "images"
        if not lbl_dir.exists():
            continue
        for lbl_file in lbl_dir.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            new_lines = []
            for line in lines:
                parts = line.split()
                if not parts:
                    continue
                orig_id = int(parts[0])
                if orig_id not in id_map:
                    continue
                parts[0] = str(id_map[orig_id])
                new_lines.append(" ".join(parts))
            if not new_lines:
                continue
            img_src = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                cand = img_dir / (lbl_file.stem + ext)
                if cand.exists():
                    img_src = cand
                    break
            if not img_src:
                continue
            new_stem = f"{out_name}_{split}_{lbl_file.stem}"
            shutil.copy2(img_src, out_img / (new_stem + img_src.suffix))
            (out_lbl / (new_stem + ".txt")).write_text(
                "\n".join(new_lines), encoding="utf-8")
            converted += 1
    print(f"     저장: {converted}개")
    return converted


print("="*60)
print(" Roboflow 다운로드 (이미 있으면 스킵)")
print("="*60)

download_results = {}
for cfg in RF_DATASETS:
    out_img = RAW_DIR / cfg["out_name"] / "images"
    if out_img.exists() and len(list(out_img.glob("*.*"))) > 50:
        cnt = len(list(out_img.glob("*.*")))
        print(f"  [SKIP] {cfg['out_name']} ({cnt}장 이미 존재)")
        download_results[cfg["out_name"]] = cnt
        continue
    print(f"\n[RF] {cfg['workspace']}/{cfg['project']}")
    try:
        proj    = rf.workspace(cfg["workspace"]).project(cfg["project"])
        dataset = proj.version(cfg["version"]).download(
            "yolov8", location=str(RAW_DIR / cfg["project"]), overwrite=False)
        n = remap_and_copy(Path(dataset.location), cfg["class_map"], cfg["out_name"])
        download_results[cfg["out_name"]] = n
    except Exception as e:
        print(f"     실패: {e}")
        download_results[cfg["out_name"]] = 0

# COCO
print("\n[COCO 2017] bicycle / car / motorcycle / person ...")
try:
    import fiftyone.zoo as foz
    COCO_MAP = {"bicycle": 0, "car": 3, "motorcycle": 4, "person": 7}
    for coco_cls, our_id in COCO_MAP.items():
        out_img = RAW_DIR / f"coco_{coco_cls}" / "images"
        out_lbl = RAW_DIR / f"coco_{coco_cls}" / "labels"
        if out_img.exists() and len(list(out_img.glob("*.*"))) >= 200:
            print(f"  [SKIP] coco_{coco_cls}")
            continue
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)
        ds = foz.load_zoo_dataset("coco-2017", split="train",
            label_types=["detections"], classes=[coco_cls],
            max_samples=600, dataset_name=f"coco_{coco_cls}", overwrite=True)
        converted = 0
        for sample in ds:
            if not sample.filepath or not Path(sample.filepath).exists():
                continue
            lines = []
            if sample.ground_truth:
                for det in sample.ground_truth.detections:
                    if det.label.lower() != coco_cls.lower():
                        continue
                    x1, y1, bw, bh = det.bounding_box
                    if bw <= 0 or bh <= 0:
                        continue
                    cx, cy = x1+bw/2, y1+bh/2
                    lines.append(f"{our_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            if not lines:
                continue
            src = Path(sample.filepath)
            shutil.copy2(src, out_img / src.name)
            (out_lbl / (src.stem + ".txt")).write_text("\n".join(lines), encoding="utf-8")
            converted += 1
        print(f"  coco_{coco_cls}: {converted}개")
except Exception as e:
    print(f"  COCO 실패: {e}")

# 요약
print("\n[클래스별 RAW 커버리지]")
raw_cnt = defaultdict(int)
for lbl_dir in RAW_DIR.rglob("labels"):
    for lf in lbl_dir.glob("*.txt"):
        for line in lf.read_text(encoding="utf-8").strip().splitlines():
            parts = line.split()
            if parts:
                try: raw_cnt[int(parts[0])] += 1
                except: pass
for cid, cname in enumerate(CLASS_NAMES):
    cnt = raw_cnt.get(cid, 0)
    flag = " ❌" if cnt == 0 else (" ⚠️" if cnt < 500 else " ✅")
    print(f"  [{cid}] {cname:<12}: {cnt:>7}개{flag}")

 Roboflow 다운로드 (이미 있으면 스킵)

[RF] esera/bollards-crosswalk-stairs
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /root/pbl2-main/data/raw/bollards-crosswalk-stairs in yolov8:: 100%|██████████| 1690/1690 [00:00<00:00, 14187.56it/s]

     원본 클래스: {0: 'Bollard', 1: 'crosswalk', 2: 'downstairs', 3: 'upstairs'}
     ID 매핑: {0: 2, 1: 6}
     저장: 463개

[RF] engs-89/bikes-pedestrians-scooters-wheelchairs
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /root/pbl2-main/data/raw/bikes-pedestrians-scooters-wheelchairs in yolov8:: 100%|██████████| 130976/130976 [00:12<00:00, 10151.78it/s]


...

     원본 클래스: {0: 'Pedestrian'}
     ID 매핑: {0: 7}
     저장: 514개

[RF] pedestriantracking-0t91d/pedestrian-detection-fqlmr
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /root/pbl2-main/data/raw/pedestrian-detection-fqlmr in yolov8:: 100%|██████████| 47510/47510 [00:04<00:00, 11132.52it/s]


     원본 클래스: {0: 'person'}
     ID 매핑: {0: 7}
     저장: 12785개

[COCO 2017] bicycle / car / motorcycle / person ...


IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Overwriting existing directory '/root/fiftyone/coco-2017/train'
Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'
 100% |██████████████████| 600/600 [1.5m elapsed, 0s remaining, 6.5 images/s]      
Writing annotations for 600 downloaded samples to '/root/fiftyone/coco-2017/train/labels.json'
Dataset info written to '/root/fiftyone/coco-2017/info.json'
Loading existing dataset 'coco_bicycle'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
  coco_bicycle: 600개
Overwriting existing directory '/root/fiftyone/coco-2017/train'
Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'
 100% |██████████████████| 600/600 [1.4m elapsed, 0s remaining, 8.3 images/s]      
Writing annotations for 600 downloaded samples to '/root/fiftyone/coco-2017/train/labels.json'
Dataset info written to '/root/fiftyone/coco-2017/info.json'
Loading existing dataset 'coco_car'. To reload from disk, either delete the ex

## Cell 2 — 병합 & 클래스 균형화

In [3]:
# ============================================================
# Cell 2: 병합 & 클래스 불균형 해소 (v7 캡 조정)
# ============================================================
MERGED_IMAGES = MERGED_DIR / "images"
MERGED_LABELS = MERGED_DIR / "labels"

if MERGED_IMAGES.exists(): shutil.rmtree(MERGED_IMAGES)
if MERGED_LABELS.exists(): shutil.rmtree(MERGED_LABELS)
MERGED_IMAGES.mkdir(parents=True, exist_ok=True)
MERGED_LABELS.mkdir(parents=True, exist_ok=True)

def merge_raw_folder(folder_name):
    base = RAW_DIR / folder_name
    candidates = [base]
    for split in ["train", "valid", "val", "test"]:
        if (base / split).exists():
            candidates.append(base / split)
    copied = 0
    for src_root in candidates:
        src_img = src_root / "images"
        src_lbl = src_root / "labels"
        if not src_lbl.exists():
            continue
        for lbl_file in src_lbl.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            if not lines:
                continue
            new_stem = f"{folder_name}_{src_root.name}_{lbl_file.stem}"
            (MERGED_LABELS / (new_stem + ".txt")).write_text(
                "\n".join(lines), encoding="utf-8")
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                img_src = src_img / (lbl_file.stem + ext)
                if img_src.exists():
                    shutil.copy2(img_src, MERGED_IMAGES / (new_stem + ext))
                    break
            copied += 1
    return copied

print("[병합 중...]")
total_merged = 0
for folder in sorted(d.name for d in RAW_DIR.iterdir() if d.is_dir()):
    n = merge_raw_folder(folder)
    print(f"  {'✅' if n > 0 else '❌'} {folder:<45}: {n}개")
    total_merged += n
print(f"\n  총 병합: {total_merged}개")

# 병합 전 분포
class_counter = defaultdict(int)
for lf in MERGED_LABELS.glob("*.txt"):
    for line in lf.read_text(encoding="utf-8").strip().splitlines():
        parts = line.split()
        if parts:
            try: class_counter[int(parts[0])] += 1
            except: pass

print("\n[캡 적용 전 클래스별 수]")
max_cnt = max(class_counter.values()) if class_counter else 1
for cid, cname in enumerate(CLASS_NAMES):
    cnt = class_counter.get(cid, 0)
    bar = "█" * int(cnt / max_cnt * 40)
    print(f"  [{cid}] {cname:<12}: {cnt:>7}개  {bar}")

# v7: 약한 클래스(bicycle/kickboard/people)는 캡을 높이고
# 강한 클래스(car)는 적당히 유지
MAX_PER_CLASS = {
    0: 10000,  # bicycle   (v6 48% → 데이터 최대 확보)
    1: 12000,  # kickboard (v6 44% → 데이터 최대 확보)
    2: 6000,   # bollard
    3: 5000,   # car
    4: 6000,   # motorcycle
    5: 6000,   # stairs
    6: 99999,  # crosswalk (전부)
    7: 12000,  # people    (v6 32% → 데이터 최대 확보)
}

print("\n[클래스 캡 적용 중...]")
random.seed(42)
all_lbls = list(MERGED_LABELS.glob("*.txt"))
random.shuffle(all_lbls)

live = defaultdict(int)
kept = removed = 0

for lbl in all_lbls:
    lines = lbl.read_text(encoding="utf-8").strip().splitlines()
    classes = []
    for l in lines:
        parts = l.split()
        if parts:
            try: classes.append(int(parts[0]))
            except: pass
    if not classes:
        lbl.unlink()
        removed += 1
        continue
    main_cls = max(set(classes), key=classes.count)
    if live[main_cls] >= MAX_PER_CLASS.get(main_cls, 99999):
        for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
            img = MERGED_IMAGES / (lbl.stem + ext)
            if img.exists():
                img.unlink()
                break
        lbl.unlink()
        removed += 1
    else:
        for c in classes:
            live[c] += 1
        kept += 1

print(f"  유지: {kept}개  제거: {removed}개")
print("\n[캡 적용 후 클래스별 수]")
for cid, cname in enumerate(CLASS_NAMES):
    cnt = live.get(cid, 0)
    bar = "█" * min(cnt // 300, 40)
    flag = " ⚠️ 부족!" if cnt < 500 else " ✅"
    print(f"  [{cid}] {cname:<12}: {cnt:>6}개  {bar}{flag}")

[병합 중...]
  ✅ bikes-pedestrians-scooters-wheelchairs       : 59858개
  ✅ bollard-3icj2                                : 1843개
  ✅ bollard-ds7rv                                : 1844개
  ✅ bollard_only_1                               : 43개
  ✅ bollard_only_2                               : 43개
  ✅ bollard_stairs_crosswalk                     : 463개
  ✅ bollards-crosswalk-stairs                    : 839개
  ✅ coco_bicycle                                 : 600개
  ✅ coco_car                                     : 600개
  ✅ coco_motorcycle                              : 600개
  ✅ coco_person                                  : 600개
  ✅ crosswalk-1elwe                              : 2650개
  ✅ crosswalk_1                                  : 240개
  ✅ escalator-stairs                             : 10444개
  ✅ kickboard-dfkjo                              : 442개
  ✅ kickboard-er5c4                              : 288개
  ✅ kickboard-ibhkj                              : 462개
  ✅ kickboard_bicycle_people     

## Cell 3 — Train/Val/Test 분할 & YAML 생성 (전체 + 클래스별)

In [4]:
# ============================================================
# Cell 3: 데이터 분할 (7:2:1) + 클래스별 서브셋 YAML 생성
# ============================================================
if PROC_DIR.exists(): shutil.rmtree(PROC_DIR)
for split in ["train", "val", "test"]:
    (PROC_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PROC_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

all_images = sorted([f for f in MERGED_IMAGES.glob("*.*")
                     if f.suffix.lower() in [".jpg",".jpeg",".png",".bmp"]])
if not all_images:
    raise RuntimeError("merged 이미지 없음")

random.seed(42)
random.shuffle(all_images)
n       = len(all_images)
n_train = int(n * 0.7)
n_val   = int(n * 0.2)
splits  = {
    "train": all_images[:n_train],
    "val":   all_images[n_train:n_train+n_val],
    "test":  all_images[n_train+n_val:],
}

split_counts = defaultdict(lambda: defaultdict(int))
for split_name, img_list in splits.items():
    for img_path in img_list:
        shutil.copy2(img_path, PROC_DIR/"images"/split_name/img_path.name)
        lbl = MERGED_LABELS / (img_path.stem + ".txt")
        if lbl.exists():
            shutil.copy2(lbl, PROC_DIR/"labels"/split_name/lbl.name)
            for line in lbl.read_text(encoding="utf-8").strip().splitlines():
                parts = line.split()
                if parts:
                    try: split_counts[split_name][int(parts[0])] += 1
                    except: pass
    print(f"  {split_name}: {len(img_list)}장")

# ---- 전체 통합 YAML ----
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.dump({
        "path":  str(PROC_DIR).replace("\\", "/"),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    NUM_CLASSES,
        "names": CLASS_NAMES,
    }, f, default_flow_style=False, allow_unicode=True)
print(f"\n전체 dataset.yaml: {CONFIG_PATH}")

# ---- 클래스별 서브셋 YAML 생성 (Phase 1용) ----
# 각 클래스별로 해당 클래스 이미지만 포함된 서브디렉토리 + YAML 생성
PER_CLASS_DATA_DIR = DATA_DIR / "per_class"
if PER_CLASS_DATA_DIR.exists():
    shutil.rmtree(PER_CLASS_DATA_DIR)

print("\n[클래스별 서브셋 생성 중...]")
CLASS_CONFIG_PATHS = {}  # cid -> yaml_path

for target_cid, target_cname in enumerate(CLASS_NAMES):
    cls_dir = PER_CLASS_DATA_DIR / target_cname
    for split in ["train", "val", "test"]:
        (cls_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (cls_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    cnt = defaultdict(int)
    for split_name, img_list in splits.items():
        for img_path in img_list:
            lbl_src = PROC_DIR / "labels" / split_name / (img_path.stem + ".txt")
            if not lbl_src.exists():
                continue
            lines = lbl_src.read_text(encoding="utf-8").strip().splitlines()
            # 해당 클래스가 포함된 이미지만 선택
            cls_lines = [l for l in lines if l.split() and int(l.split()[0]) == target_cid]
            if not cls_lines:
                continue
            # 이미지 복사
            img_src = PROC_DIR / "images" / split_name / img_path.name
            if not img_src.exists():
                continue
            shutil.copy2(img_src, cls_dir / "images" / split_name / img_path.name)
            # 라벨은 해당 클래스만 포함 (다중 클래스 이미지에서 해당 클래스만 남김)
            (cls_dir / "labels" / split_name / lbl_src.name).write_text(
                "\n".join(cls_lines), encoding="utf-8")
            cnt[split_name] += 1

    # 클래스별 YAML (단일 클래스)
    cls_yaml_path = CONFIG_DIR / f"dataset_{target_cname}.yaml"
    with open(cls_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump({
            "path":  str(cls_dir).replace("\\", "/"),
            "train": "images/train",
            "val":   "images/val",
            "test":  "images/test",
            "nc":    NUM_CLASSES,         # 전체 클래스 수 유지 (Phase 2 호환)
            "names": CLASS_NAMES,
        }, f, default_flow_style=False, allow_unicode=True)
    CLASS_CONFIG_PATHS[target_cid] = cls_yaml_path
    total_cls = sum(cnt.values())
    print(f"  [{target_cid}] {target_cname:<12}: train={cnt['train']} val={cnt['val']} test={cnt['test']} (총 {total_cls}장)")

print("\n[클래스 분포 (전체)]")
header = f"{'Class':<14}" + "".join(f"{s:>8}" for s in ["train","val","test","total"])
print("  " + header)
print("  " + "-"*len(header))
for cid, cname in enumerate(CLASS_NAMES):
    tr  = split_counts["train"].get(cid, 0)
    va  = split_counts["val"].get(cid, 0)
    te  = split_counts["test"].get(cid, 0)
    tot = tr+va+te
    flag = " ❌" if tot == 0 else (" ⚠️" if tot < 200 else " ✅")
    print(f"  [{cid}] {cname:<12}" + f"{tr:>8}{va:>8}{te:>8}{tot:>8}{flag}")

  train: 20695장
  val: 5913장
  test: 2957장

전체 dataset.yaml: /root/pbl2-main/configs/dataset_8class.yaml

[클래스별 서브셋 생성 중...]
  [0] bicycle     : train=3134 val=905 test=415 (총 4454장)
  [1] kickboard   : train=4519 val=1282 test=684 (총 6485장)
  [2] bollard     : train=3184 val=819 test=452 (총 4455장)
  [3] car         : train=2539 val=746 test=342 (총 3627장)
  [4] motorcycle  : train=1905 val=565 test=291 (총 2761장)
  [5] stairs      : train=2681 val=729 test=373 (총 3783장)
  [6] crosswalk   : train=595 val=182 test=85 (총 862장)
  [7] people      : train=3055 val=964 test=459 (총 4478장)

[클래스 분포 (전체)]
  Class            train     val    test   total
  ----------------------------------------------
  [0] bicycle         7120    2011     968   10099 ✅
  [1] kickboard       8408    2441    1240   12089 ✅
  [2] bollard         4285    1112     620    6017 ✅
  [3] car             3487    1028     488    5003 ✅
  [4] motorcycle      4229    1225     555    6009 ✅
  [5] stairs          4261    1172 

## Cell 4 — Phase 1: 클래스별 개별 학습 (YOLOv8x)

In [5]:
# ============================================================
# Cell 4: Phase 1 — 클래스별 개별 학습
#
# 전략:
#   - 각 클래스에 특화된 데이터로 YOLOv8x를 짧게 학습 (50 epoch)
#   - 성능이 낮은 클래스(bicycle, kickboard, people)를 먼저 집중 학습
#   - 각 클래스 best.pt 저장 → Phase 2에서 앙상블 or 가중치 초기화로 활용
# ============================================================
import torch

# v7 클래스별 학습 설정
# 성능이 낮은 클래스에 더 많은 epoch 할당
PER_CLASS_EPOCHS = {
    0: 60,  # bicycle   (v6 48% → 집중 학습)
    1: 60,  # kickboard (v6 44% → 집중 학습)
    2: 40,  # bollard   (v6 73%)
    3: 30,  # car       (v6 85% → 짧게)
    4: 45,  # motorcycle(v6 65%)
    5: 45,  # stairs    (v6 70%)
    6: 40,  # crosswalk (v6 73%)
    7: 60,  # people    (v6 32% → 집중 학습)
}

# 클래스별 augmentation 강도 (어려운 클래스는 더 강하게)
PER_CLASS_AUG = {
    # bicycle: 작고 빠름 → scale/translate 강화
    0: {"scale": 0.7, "translate": 0.15, "mosaic": 1.0, "copy_paste": 0.4},
    # kickboard: bicycle과 혼동 → 강한 augmentation
    1: {"scale": 0.7, "translate": 0.15, "mosaic": 1.0, "copy_paste": 0.4},
    # bollard: 정적 객체 → 기본
    2: {"scale": 0.5, "translate": 0.1,  "mosaic": 1.0, "copy_paste": 0.2},
    # car: 이미 잘 됨 → 기본
    3: {"scale": 0.5, "translate": 0.1,  "mosaic": 1.0, "copy_paste": 0.2},
    # motorcycle: bicycle과 혼동 → 강화
    4: {"scale": 0.6, "translate": 0.12, "mosaic": 1.0, "copy_paste": 0.3},
    # stairs: 질감 중요 → hsv 강화
    5: {"scale": 0.5, "translate": 0.1,  "mosaic": 1.0, "copy_paste": 0.2, "hsv_s": 0.9},
    # crosswalk: 패턴 중요
    6: {"scale": 0.5, "translate": 0.1,  "mosaic": 1.0, "copy_paste": 0.2},
    # people: 다양한 포즈 → 강한 augmentation
    7: {"scale": 0.7, "translate": 0.15, "mosaic": 1.0, "copy_paste": 0.5, "degrees": 15.0},
}

BASE_TRAIN_CONFIG = {
    "device":        DEVICE,
    "amp":           True,
    "workers":       0,
    "batch":         BATCH,
    "imgsz":         640,
    "patience":      15,
    "fraction":      1.0,
    "cache":         False,
    "optimizer":     "AdamW",
    "lr0":           0.001,
    "lrf":           0.01,
    "weight_decay":  0.0005,
    "warmup_epochs": 3,
    "cos_lr":        True,
    "mosaic":        1.0,
    "mixup":         0.0,
    "close_mosaic":  10,
    "copy_paste":    0.2,
    "degrees":       10.0,
    "translate":     0.1,
    "scale":         0.5,
    "shear":         2.0,
    "perspective":   0.0,
    "flipud":        0.0,
    "fliplr":        0.5,
    "hsv_h":         0.015,
    "hsv_s":         0.7,
    "hsv_v":         0.4,
    "erasing":       0.3,
    "cls":           1.5,
    "box":           7.5,
    "dfl":           1.5,
    "save":          True,
    "save_period":   10,
    "plots":         True,
    "verbose":       True,
}

print("="*60)
print(" Phase 1: 클래스별 개별 학습 (YOLOv8x)")
print("="*60)

phase1_results = {}

# 성능이 낮은 클래스부터 학습 (7→0→1→4→5→6→2→3)
TRAIN_ORDER = [7, 0, 1, 4, 5, 6, 2, 3]  # people → bicycle → kickboard → ...

for cid in TRAIN_ORDER:
    cname = CLASS_NAMES[cid]
    cls_yaml = CLASS_CONFIG_PATHS.get(cid)
    if cls_yaml is None or not cls_yaml.exists():
        print(f"\n[SKIP] {cname}: YAML 없음")
        continue

    save_dir = PHASE1_DIR / cname
    best_pt  = save_dir / "weights" / "best.pt"
    if best_pt.exists():
        print(f"\n[SKIP] {cname}: 이미 학습됨 ({best_pt})")
        phase1_results[cid] = str(best_pt)
        continue

    print(f"\n{'='*50}")
    print(f" [{cid}] {cname} 학습 시작 ({PER_CLASS_EPOCHS[cid]} epochs)")
    print(f"{'='*50}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 클래스별 aug 설정 병합
    cfg = {**BASE_TRAIN_CONFIG}
    cfg.update(PER_CLASS_AUG.get(cid, {}))
    cfg["data"]     = str(cls_yaml)
    cfg["epochs"]   = PER_CLASS_EPOCHS[cid]
    cfg["project"]  = str(PHASE1_DIR)
    cfg["name"]     = cname
    cfg["exist_ok"] = True

    try:
        model = YOLO("yolov8x.pt")  # ← YOLOv8x (최대 모델)
        result = model.train(**cfg)
        phase1_results[cid] = str(best_pt)
        print(f"  ✅ {cname} 학습 완료 → {best_pt}")
    except Exception as e:
        print(f"  ❌ {cname} 학습 실패: {e}")
        phase1_results[cid] = None

print("\n" + "="*60)
print(" Phase 1 완료 요약")
print("="*60)
for cid, pt_path in sorted(phase1_results.items()):
    status = "✅" if pt_path and Path(pt_path).exists() else "❌"
    print(f"  {status} [{cid}] {CLASS_NAMES[cid]:<12}: {pt_path}")

 Phase 1: 클래스별 개별 학습 (YOLOv8x)

 [7] people 학습 시작 (60 epochs)
Ultralytics 8.4.42 🚀 Python-3.10.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/root/pbl2-main/configs/dataset_people.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.3, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=people, nbs=64

## Cell 5 — Phase 1 중간 평가

In [6]:
# ============================================================
# Cell 5: Phase 1 클래스별 모델 중간 평가
# 각 클래스 모델을 전체 테스트셋에서 평가하여 개선 확인
# ============================================================
print("="*60)
print(" Phase 1 클래스별 모델 평가")
print("="*60)

phase1_ap = {}  # cid -> ap50

for cid in range(NUM_CLASSES):
    cname = CLASS_NAMES[cid]
    best_pt = PHASE1_DIR / cname / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"  [{cid}] {cname:<12}: ❌ best.pt 없음")
        continue

    try:
        m = YOLO(str(best_pt))
        # 클래스별 전용 테스트셋으로 평가
        cls_yaml = CLASS_CONFIG_PATHS.get(cid)
        if cls_yaml and cls_yaml.exists():
            metrics = m.val(data=str(cls_yaml), split="test",
                            device=DEVICE, batch=BATCH, workers=0, verbose=False)
            if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
                ap_vals = metrics.box.ap50
                # 해당 클래스 AP 찾기
                ap = float(ap_vals[cid]) if cid < len(ap_vals) else 0.0
            else:
                ap = float(metrics.box.map50)
            phase1_ap[cid] = ap
            v6_ap = [0.480, 0.445, 0.737, 0.858, 0.656, 0.708, 0.736, 0.329][cid]
            diff  = ap - v6_ap
            arrow = "↑" if diff > 0 else "↓"
            flag  = "✅" if ap >= 0.8 else ("⚠️" if ap >= 0.6 else "❌")
            print(f"  {flag} [{cid}] {cname:<12}: {ap*100:.1f}%  (v6: {v6_ap*100:.1f}% {arrow}{abs(diff)*100:.1f}%)")
    except Exception as e:
        print(f"  [{cid}] {cname:<12}: 평가 실패 ({e})")

print("\n→ Phase 2 (통합 파인튜닝)으로 이동하세요")

 Phase 1 클래스별 모델 평가
Ultralytics 8.4.42 🚀 Python-3.10.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 113 layers, 68,131,272 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 30.9±6.0 MB/s, size: 63.4 KB)
val: Scanning /root/pbl2-main/data/per_class/bicycle/labels/test... 415 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 415/415 1.4Kit/s 0.3s<0.1s
val: New cache created: /root/pbl2-main/data/per_class/bicycle/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 491, len(boxes) = 968. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 4.1it/s 6.4s0.2s
                   all        415        968      0.813      0.647      0.733      

## Cell 6 — Phase 2: 통합 파인튜닝 (YOLOv8x)

In [ ]:
# ============================================================
# Cell 6: Phase 2 — 통합 파인튜닝
#
# 전략:
#   - Phase 1에서 가장 성능이 좋은 클래스 모델의 가중치를 초기값으로 사용
#     (없으면 yolov8x.pt 사용)
#   - 전체 8클래스 통합 데이터로 파인튜닝 (150 epoch)
#   - lr0 낮춤 (파인튜닝이므로 학습률 작게)
# ============================================================
import torch

# Phase 1에서 가장 성능 좋은 모델 선택 (또는 최고 mAP 모델)
# 기본적으로 people 모델 (가장 어렵고, 다양한 특징 학습) 또는 yolov8x.pt
init_model_path = None

# Phase 1 결과 중 가장 AP가 높은 클래스 모델을 초기값으로
if phase1_ap:
    best_cid = max(phase1_ap, key=phase1_ap.get)
    candidate = PHASE1_DIR / CLASS_NAMES[best_cid] / "weights" / "best.pt"
    if candidate.exists():
        init_model_path = str(candidate)
        print(f"Phase 1 최고 모델: [{best_cid}] {CLASS_NAMES[best_cid]} (AP={phase1_ap[best_cid]*100:.1f}%)")
        print(f"초기 가중치: {init_model_path}")

if not init_model_path:
    # people 모델 우선 시도
    people_pt = PHASE1_DIR / "people" / "weights" / "best.pt"
    if people_pt.exists():
        init_model_path = str(people_pt)
        print(f"초기 가중치: people Phase1 모델")
    else:
        init_model_path = "yolov8x.pt"
        print(f"초기 가중치: yolov8x.pt (pretrained)")

PHASE2_NAME = "safewalk_v7"
best_pt_v7  = PHASE2_DIR / "weights" / "best.pt"

PHASE2_CONFIG = {
    "data":          str(CONFIG_PATH),
    "device":        DEVICE,
    "amp":           True,
    "workers":       0,

    # 학습 설정
    "epochs":        150,
    "batch":         BATCH,
    "imgsz":         640,
    "patience":      0,       # 얼리스탑 없음 (150 epoch 전부 학습)
    "fraction":      1.0,
    "cache":         False,

    # 옵티마이저 (파인튜닝 → lr 낮춤)
    "optimizer":     "AdamW",
    "lr0":           0.0005,  # Phase 1보다 낮게 (파인튜닝)
    "lrf":           0.005,
    "weight_decay":  0.0005,
    "warmup_epochs": 5,
    "cos_lr":        True,

    # 증강
    "mosaic":        1.0,
    "mixup":         0.05,    # 통합 학습에서 약하게 mixup 허용
    "close_mosaic":  20,
    "copy_paste":    0.3,
    "degrees":       10.0,
    "translate":     0.1,
    "scale":         0.6,
    "shear":         2.0,
    "perspective":   0.0,
    "flipud":        0.0,
    "fliplr":        0.5,
    "hsv_h":         0.015,
    "hsv_s":         0.7,
    "hsv_v":         0.4,
    "erasing":       0.3,

    # Loss 가중치
    "cls":           2.0,    # 클래스 구분 더 강화 (v6: 1.5 → v7: 2.0)
    "box":           7.5,
    "dfl":           1.5,

    # 저장
    "project":       str(MODELS_DIR),
    "name":          PHASE2_NAME,
    "exist_ok":      True,
    "save":          True,
    "save_period":   10,
    "plots":         True,
    "verbose":       True,
}

sep = "="*60
print(sep)
print(" Phase 2: 통합 파인튜닝 (YOLOv8x)")
print(sep)
print(f" 초기 모델 : {init_model_path}")
print(f" imgsz     : 640")
print(f" Epochs    : 150  patience=0")
print(f" lr0       : 0.0005 (파인튜닝)")
print(f" cls loss  : 2.0")
print(sep)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

RESUME = False
last_ckpt = PHASE2_DIR / "weights" / "last.pt"

if RESUME and last_ckpt.exists():
    print(f"[RESUME] {last_ckpt}")
    model = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    model = YOLO(init_model_path)
    results = model.train(**PHASE2_CONFIG)

print(f"\n✅ Phase 2 완료! → {PHASE2_DIR}/weights/best.pt")

Phase 1 최고 모델: [0] bicycle (AP=73.3%)
초기 가중치: /root/pbl2-main/models/phase1_per_class/bicycle/weights/best.pt
 Phase 2: 통합 파인튜닝 (YOLOv8x)
 초기 모델 : /root/pbl2-main/models/phase1_per_class/bicycle/weights/best.pt
 imgsz     : 640
 Epochs    : 150  patience=0
 lr0       : 0.0005 (파인튜닝)
 cls loss  : 2.0
Ultralytics 8.4.42 🚀 Python-3.10.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/root/pbl2-main/configs/dataset_8class.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.3, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0

## Cell 7 — 최종 평가

In [ ]:
# ============================================================
# Cell 7: 테스트셋 최종 평가
# ============================================================
best_pt = PHASE2_DIR / "weights" / "best.pt"
if not best_pt.exists():
    print(f"[ERROR] {best_pt} 없음")
else:
    eval_model = YOLO(str(best_pt))
    metrics = eval_model.val(
        data=str(CONFIG_PATH), split="test",
        device=DEVICE, batch=BATCH, workers=0, verbose=True,
    )
    print("\n" + "="*60)
    print(" [최종 Test Results — SafeWalk v7]")
    print("="*60)
    print(f"  mAP@0.5:      {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print(f"  Precision:    {metrics.box.mp:.4f}")
    print(f"  Recall:       {metrics.box.mr:.4f}")

    V6_AP = [0.4799, 0.4450, 0.7369, 0.8585, 0.6555, 0.7082, 0.7362, 0.3288]
    print("\n[클래스별 AP@0.5 — v6 대비 비교]")
    print(f"  {'Class':<12} {'v7':>8} {'v6':>8} {'diff':>8}")
    print(f"  {'-'*40}")
    if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
        for cid, cname in enumerate(CLASS_NAMES):
            if cid < len(metrics.box.ap50):
                ap   = float(metrics.box.ap50[cid])
                v6   = V6_AP[cid]
                diff = ap - v6
                flag = "✅" if ap >= 0.8 else ("⚠️" if ap >= 0.6 else "❌")
                arrow = "↑" if diff > 0 else "↓"
                print(f"  {flag} {cname:<12} {ap*100:>7.1f}% {v6*100:>7.1f}% {arrow}{abs(diff)*100:>6.1f}%")

## Cell 8 — 실시간 처리 최적화 (TensorRT Export & 속도 측정)

In [ ]:
# ============================================================
# Cell 8: 실시간 처리 최적화
#
# 1. TensorRT 엔진 export (GPU 가속)
# 2. FP16 half precision export
# 3. 추론 속도(FPS) 벤치마크
# 4. 실시간 추론 함수 제공
# ============================================================
import time
import numpy as np

best_pt = PHASE2_DIR / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(f"{best_pt} 없음 — Cell 6 먼저 실행")

print("="*60)
print(" 실시간 처리 최적화")
print("="*60)

# ---- 1. TensorRT Export (GPU 있을 때만) ----
trt_path = PHASE2_DIR / "weights" / "best.engine"
fp16_path = PHASE2_DIR / "weights" / "best_fp16.pt"

if DEVICE != "cpu" and torch.cuda.is_available():
    if not trt_path.exists():
        print("\n[1] TensorRT 엔진 Export 중... (시간 소요)")
        try:
            export_model = YOLO(str(best_pt))
            export_model.export(
                format="engine",
                imgsz=640,
                half=True,        # FP16
                device=0,
                workspace=4,       # GB
                verbose=False,
            )
            print(f"  ✅ TensorRT 엔진 저장: {trt_path}")
        except Exception as e:
            print(f"  ⚠️ TensorRT Export 실패 (tensorrt 설치 필요): {e}")
            print("  → pip install tensorrt 후 재시도")
    else:
        print(f"\n[1] TensorRT 엔진 이미 존재: {trt_path}")
else:
    print("\n[1] CPU 모드 — TensorRT 스킵")

# ---- 2. FPS 벤치마크 ----
print("\n[2] 추론 속도 벤치마크")

def benchmark_fps(model_path, n_warmup=10, n_test=100, imgsz=640):
    """모델 추론 FPS 측정"""
    model = YOLO(str(model_path))
    dummy = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

    # 워밍업
    for _ in range(n_warmup):
        model.predict(dummy, verbose=False, device=DEVICE)

    if DEVICE != "cpu" and torch.cuda.is_available():
        torch.cuda.synchronize()

    # 속도 측정
    start = time.perf_counter()
    for _ in range(n_test):
        model.predict(dummy, verbose=False, device=DEVICE)
    if DEVICE != "cpu" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    fps = n_test / elapsed
    ms  = elapsed / n_test * 1000
    return fps, ms

# PyTorch 모델 속도
print(f"  PyTorch (FP32):")
try:
    fps, ms = benchmark_fps(best_pt)
    print(f"    FPS: {fps:.1f}  ({ms:.1f} ms/frame)")
    print(f"    {'✅ 실시간 가능 (30+ FPS)' if fps >= 30 else '⚠️ 실시간 어려움 — TensorRT 권장'}")
except Exception as e:
    print(f"    측정 실패: {e}")

# TensorRT 모델 속도
if trt_path.exists():
    print(f"\n  TensorRT (FP16):")
    try:
        fps_trt, ms_trt = benchmark_fps(trt_path)
        print(f"    FPS: {fps_trt:.1f}  ({ms_trt:.1f} ms/frame)")
        print(f"    {'✅ 실시간 가능 (30+ FPS)' if fps_trt >= 30 else '⚠️ 실시간 어려움'}")
    except Exception as e:
        print(f"    측정 실패: {e}")

# ---- 3. 실시간 추론 클래스 ----
print("\n[3] 실시간 추론 설정 저장")

# 실시간 처리를 위한 최적 설정
REALTIME_CONFIG = {
    "conf":      0.4,    # confidence threshold (낮추면 recall↑, 높이면 precision↑)
    "iou":       0.45,   # NMS IOU threshold
    "imgsz":     640,
    "half":      True if DEVICE != "cpu" else False,  # FP16 (GPU만)
    "device":    DEVICE,
    "max_det":   50,     # 최대 탐지 수 (실시간에서 제한)
    "verbose":   False,
    "stream":    True,   # 스트리밍 모드 (메모리 효율)
}

import json
rt_config_path = PHASE2_DIR / "realtime_config.json"
with open(rt_config_path, "w") as f:
    json.dump(REALTIME_CONFIG, f, indent=2)
print(f"  실시간 설정 저장: {rt_config_path}")
print(f"  설정 내용:")
for k, v in REALTIME_CONFIG.items():
    print(f"    {k}: {v}")

print("\n" + "="*60)
print(" 실시간 추론 사용 예시 (Cell 9 참고)")
print("="*60)

## Cell 9 — 실시간 추론 데모

In [ ]:
# ============================================================
# Cell 9: 실시간 추론 데모
#
# 웹캠 / 동영상 파일 / 이미지 폴더에 대해
# 실시간 추론 + FPS 표시 + 결과 저장
# ============================================================
import cv2, time, json
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# ---- 설정 ----
# TensorRT 엔진이 있으면 사용, 없으면 PyTorch 모델
trt_path = PHASE2_DIR / "weights" / "best.engine"
best_pt  = PHASE2_DIR / "weights" / "best.pt"

MODEL_PATH = str(trt_path) if trt_path.exists() else str(best_pt)
print(f"사용 모델: {MODEL_PATH}")

# 실시간 설정 로드
rt_config_path = PHASE2_DIR / "realtime_config.json"
if rt_config_path.exists():
    with open(rt_config_path) as f:
        RT = json.load(f)
else:
    RT = {"conf": 0.4, "iou": 0.45, "imgsz": 640, "half": False,
          "device": DEVICE, "max_det": 50, "verbose": False}

# ---- 클래스별 색상 (BGR) ----
CLASS_COLORS = [
    (0,   165, 255),  # 0 bicycle   - 주황
    (255, 0,   255),  # 1 kickboard - 마젠타
    (0,   255, 255),  # 2 bollard   - 노란
    (0,   0,   255),  # 3 car       - 빨강
    (255, 128, 0),    # 4 motorcycle- 파랑+주황
    (128, 0,   255),  # 5 stairs    - 보라
    (0,   255, 0),    # 6 crosswalk - 초록
    (255, 255, 0),    # 7 people    - 하늘
]

CLASS_KO = ['자전거', '킥보드', '볼라드', '차량',
            '오토바이', '계단', '횡단보도', '사람']

# 위험도 레벨 (보행 안전 기준)
DANGER_LEVEL = {
    0: "⚠️ 위험",   # bicycle
    1: "⚠️ 위험",   # kickboard
    2: "ℹ️ 주의",   # bollard
    3: "🚨 차량",   # car
    4: "⚠️ 위험",   # motorcycle
    5: "ℹ️ 주의",   # stairs
    6: "✅ 안전",   # crosswalk
    7: "👤 보행자", # people
}


class SafeWalkDetector:
    """SafeWalk 실시간 탐지기"""

    def __init__(self, model_path, rt_config):
        self.model   = YOLO(model_path)
        self.rt      = rt_config
        self.fps_buf = []
        self.prev_t  = time.perf_counter()

    def predict_frame(self, frame):
        """단일 프레임 추론 (실시간용)"""
        results = self.model.predict(
            frame,
            conf=self.rt.get("conf", 0.4),
            iou=self.rt.get("iou", 0.45),
            imgsz=self.rt.get("imgsz", 640),
            half=self.rt.get("half", False),
            device=self.rt.get("device", "cpu"),
            max_det=self.rt.get("max_det", 50),
            verbose=False,
        )
        return results[0]

    def draw_results(self, frame, result):
        """탐지 결과를 프레임에 그리기"""
        h, w = frame.shape[:2]

        # FPS 계산
        now = time.perf_counter()
        fps = 1.0 / max(now - self.prev_t, 1e-6)
        self.prev_t = now
        self.fps_buf.append(fps)
        if len(self.fps_buf) > 30:
            self.fps_buf.pop(0)
        avg_fps = np.mean(self.fps_buf)

        # 바운딩 박스 그리기
        detected = []
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                cid  = int(box.cls[0])
                conf = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                color = CLASS_COLORS[cid] if cid < len(CLASS_COLORS) else (255,255,255)

                # 박스
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                # 레이블
                label = f"{CLASS_KO[cid]} {conf:.0%}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
                cv2.putText(frame, label, (x1+3, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

                detected.append((cid, conf))

        # HUD (좌상단)
        cv2.rectangle(frame, (0, 0), (260, 55), (0,0,0), -1)
        cv2.putText(frame, f"SafeWalk v7 | FPS: {avg_fps:.0f}",
                    (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 1)
        cv2.putText(frame, f"탐지: {len(detected)}개 객체",
                    (8, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

        # 위험도 알림 (우상단)
        danger_cls = [cid for cid, _ in detected if cid in [0,1,3,4]]
        if danger_cls:
            alert_txt = f"주의: {', '.join(CLASS_KO[c] for c in set(danger_cls))}"
            cv2.rectangle(frame, (w-300, 0), (w, 35), (0,0,180), -1)
            cv2.putText(frame, alert_txt, (w-295, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)

        return frame, avg_fps, detected

    def run_webcam(self, cam_id=0, save_path=None):
        """웹캠 실시간 추론"""
        cap = cv2.VideoCapture(cam_id)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        cap.set(cv2.CAP_PROP_FPS, 30)

        writer = None
        if save_path:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(save_path, fourcc, 30, (1280, 720))

        print("[웹캠 시작] 'q' 키로 종료")
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            if writer:
                writer.write(frame)
            cv2.imshow("SafeWalk v7 Real-time", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()
        print("종료")

    def run_video(self, video_path, save_path=None, show=False):
        """동영상 파일 추론 + 저장"""
        cap = cv2.VideoCapture(str(video_path))
        fps_orig = cap.get(cv2.CAP_PROP_FPS) or 30
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        writer = None
        if save_path:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(save_path), fourcc, fps_orig, (w, h))

        frame_idx = 0
        fps_list  = []
        print(f"[동영상 추론] {video_path} | {total}프레임")

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            fps_list.append(fps)

            if writer:
                writer.write(frame)
            if show:
                cv2.imshow("SafeWalk v7", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            frame_idx += 1
            if frame_idx % 50 == 0:
                print(f"  {frame_idx}/{total} | 평균 FPS: {np.mean(fps_list):.1f}")

        cap.release()
        if writer:
            writer.release()
        if show:
            cv2.destroyAllWindows()

        avg_fps = np.mean(fps_list) if fps_list else 0
        print(f"\n완료: {frame_idx}프레임 | 평균 FPS: {avg_fps:.1f}")
        if save_path:
            print(f"저장: {save_path}")
        return avg_fps

    def run_image_folder(self, img_dir, save_dir=None):
        """이미지 폴더 일괄 추론"""
        img_dir  = Path(img_dir)
        imgs     = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        if save_dir:
            Path(save_dir).mkdir(parents=True, exist_ok=True)

        print(f"[이미지 추론] {len(imgs)}장")
        for i, img_path in enumerate(imgs):
            frame  = cv2.imread(str(img_path))
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            if save_dir:
                cv2.imwrite(str(Path(save_dir) / img_path.name), frame)
            if (i+1) % 10 == 0:
                print(f"  {i+1}/{len(imgs)} 처리 완료")
        print("완료")


# ---- 인스턴스 생성 ----
detector = SafeWalkDetector(MODEL_PATH, RT)
print("\n✅ SafeWalkDetector 준비 완료")
print("\n사용법:")
print("  # 웹캠")
print("  detector.run_webcam(cam_id=0)")
print("")
print("  # 동영상")
print("  detector.run_video('input.mp4', save_path='output.mp4')")
print("")
print("  # 이미지 폴더")
print("  detector.run_image_folder('/path/to/images', save_dir='/path/to/output')")
print("")
print("  # 단일 이미지 추론")
print("  import cv2")
print("  frame = cv2.imread('test.jpg')")
print("  result = detector.predict_frame(frame)")
print("  frame, fps, dets = detector.draw_results(frame, result)")

## Cell 10 — 시각화 (12개 그래프)

In [ ]:
# ============================================================
# Cell 10: 시각화 (12개 그래프, 한글 지원)
# ============================================================
import subprocess
subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], check=False)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import seaborn as sns

# 한글 폰트
fm._load_fontmanager(try_read_cache=False)
nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f or 'nanum' in f]
if nanum:
    font_prop = fm.FontProperties(fname=nanum[0])
    matplotlib.rc('font', family=font_prop.get_name())
    print(f'한글 폰트: {font_prop.get_name()}')
else:
    matplotlib.rc('font', family='DejaVu Sans')
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

RESULTS_DIR = PHASE2_DIR
CSV_PATH    = RESULTS_DIR / 'results.csv'
SAVE_DIR    = RESULTS_DIR / 'plots_custom'
SAVE_DIR.mkdir(exist_ok=True)

if not CSV_PATH.exists():
    print(f'[ERROR] {CSV_PATH} 없음 — Cell 6 완료 후 실행')
else:
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    print(f'results.csv: {len(df)} epochs')

    epochs = df['epoch'].values if 'epoch' in df.columns else np.arange(len(df))

    COL = {}
    col_map = {
        'train_box': ['train/box_loss', 'train/box_om'],
        'train_cls': ['train/cls_loss', 'train/cls_om'],
        'train_dfl': ['train/dfl_loss', 'train/dfl_om'],
        'val_box':   ['val/box_loss',   'val/box_om'],
        'val_cls':   ['val/cls_loss',   'val/cls_om'],
        'val_dfl':   ['val/dfl_loss',   'val/dfl_om'],
        'precision': ['metrics/precision(B)', 'metrics/precision'],
        'recall':    ['metrics/recall(B)',    'metrics/recall'],
        'map50':     ['metrics/mAP50(B)',     'metrics/mAP_0.5'],
        'map50_95':  ['metrics/mAP50-95(B)',  'metrics/mAP_0.5:0.95'],
        'lr':        ['lr/pg0', 'lr0'],
    }
    for key, candidates in col_map.items():
        for c in candidates:
            if c in df.columns:
                COL[key] = c
                break

    CLASS_KO = ['자전거', '킥보드', '볼라드', '차량',
                '오토바이', '계단', '횡단보도', '사람']
    V6_AP    = [0.480, 0.445, 0.737, 0.858, 0.656, 0.708, 0.736, 0.329]

    def make_ax(ax, title, xlabel='Epoch', ylabel=''):
        ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    def best_marker(ax, epochs, vals, color='red'):
        idx = vals.idxmax()
        ax.axvline(epochs[idx], color=color, linestyle=':', alpha=0.8, lw=1.5)
        ax.annotate(f'최고\n{vals.max()*100:.1f}%\n(ep {int(epochs[idx])})',
                    xy=(epochs[idx], vals.max()),
                    xytext=(epochs[idx]+len(epochs)*0.07, vals.max()-0.1),
                    fontsize=8, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

    fig = plt.figure(figsize=(26, 36))
    fig.patch.set_facecolor('white')
    fig.suptitle('SafeWalk v7 학습 결과 종합 리포트\n(YOLOv8x | Phase1 클래스별 + Phase2 통합 파인튜닝)',
                 fontsize=18, fontweight='bold', y=0.99)

    ax1 = fig.add_subplot(4, 3, 1)
    for k, lbl, c in [('train_box','Box','#4C72B0'),('train_cls','Cls','#DD8452'),('train_dfl','DFL','#55A868')]:
        if k in COL: ax1.plot(epochs, df[COL[k]], color=c, lw=2, label=lbl)
    make_ax(ax1, '① Train Loss 3종', ylabel='Loss')
    ax1.legend(fontsize=9)

    ax2 = fig.add_subplot(4, 3, 2)
    for k, lbl, c in [('val_box','Box','#4C72B0'),('val_cls','Cls','#DD8452'),('val_dfl','DFL','#55A868')]:
        if k in COL: ax2.plot(epochs, df[COL[k]], color=c, lw=2, label=lbl)
    make_ax(ax2, '② Val Loss 3종', ylabel='Loss')
    ax2.legend(fontsize=9)

    ax3 = fig.add_subplot(4, 3, 3)
    if 'train_box' in COL: ax3.plot(epochs, df[COL['train_box']], '#4C72B0', lw=2, label='Train')
    if 'val_box'   in COL: ax3.plot(epochs, df[COL['val_box']],   '#C44E52', lw=2, ls='--', label='Val')
    make_ax(ax3, '③ Train vs Val Box Loss', ylabel='Loss')
    ax3.legend(fontsize=9)

    ax4 = fig.add_subplot(4, 3, 4)
    if 'train_cls' in COL: ax4.plot(epochs, df[COL['train_cls']], '#DD8452', lw=2, label='Train')
    if 'val_cls'   in COL: ax4.plot(epochs, df[COL['val_cls']],   '#C44E52', lw=2, ls='--', label='Val')
    make_ax(ax4, '④ Train vs Val Cls Loss', ylabel='Loss')
    ax4.legend(fontsize=9)

    ax5 = fig.add_subplot(4, 3, 5)
    if 'map50' in COL:
        vals = df[COL['map50']]
        ax5.plot(epochs, vals, '#4C72B0', lw=2.5)
        ax5.fill_between(epochs, vals, alpha=0.12, color='#4C72B0')
        best_marker(ax5, epochs, vals)
        ax5.axhline(0.8, color='green', ls='--', alpha=0.4, lw=1)
        ax5.text(epochs[0], 0.81, '목표 80%', fontsize=8, color='green', alpha=0.7)
    make_ax(ax5, '⑤ mAP@0.5 추이', ylabel='mAP@0.5')
    ax5.set_ylim(0, 1.05)

    ax6 = fig.add_subplot(4, 3, 6)
    if 'map50_95' in COL:
        vals = df[COL['map50_95']]
        ax6.plot(epochs, vals, '#8172B2', lw=2.5)
        ax6.fill_between(epochs, vals, alpha=0.12, color='#8172B2')
        best_marker(ax6, epochs, vals, color='#8172B2')
    make_ax(ax6, '⑥ mAP@0.5:0.95 추이', ylabel='mAP@0.5:0.95')
    ax6.set_ylim(0, 1.05)

    ax7 = fig.add_subplot(4, 3, 7)
    if 'map50'    in COL: ax7.plot(epochs, df[COL['map50']],    '#4C72B0', lw=2.5, label='mAP@0.5')
    if 'map50_95' in COL: ax7.plot(epochs, df[COL['map50_95']],'#8172B2', lw=2.5, ls='--', label='mAP@0.5:0.95')
    make_ax(ax7, '⑦ mAP@0.5 vs mAP@0.5:0.95', ylabel='mAP')
    ax7.set_ylim(0, 1.05)
    ax7.legend(fontsize=9)

    ax8 = fig.add_subplot(4, 3, 8)
    if 'precision' in COL: ax8.plot(epochs, df[COL['precision']], '#55A868', lw=2, label='Precision')
    if 'recall'    in COL: ax8.plot(epochs, df[COL['recall']],    '#C44E52', lw=2, label='Recall')
    make_ax(ax8, '⑧ Precision & Recall', ylabel='Score')
    ax8.set_ylim(0, 1.05)
    ax8.legend(fontsize=9)

    # ⑨ v6 vs v7 클래스별 AP 비교 막대
    ax9 = fig.add_subplot(4, 3, 9)
    ap50_vals = None
    best_pt_path = RESULTS_DIR / 'weights' / 'best.pt'
    if best_pt_path.exists():
        try:
            _m   = YOLO(str(best_pt_path))
            _met = _m.val(data=str(CONFIG_PATH), split='test',
                          device=DEVICE, batch=BATCH, workers=0, verbose=False)
            if hasattr(_met.box, 'ap50') and _met.box.ap50 is not None:
                ap50_vals = [float(v) for v in _met.box.ap50[:8]]
        except Exception as e:
            print(f'AP 재계산 실패: {e}')

    x = np.arange(len(CLASS_KO))
    w = 0.35
    ax9.bar(x - w/2, V6_AP,      w, label='v6', color='#C44E52', alpha=0.7, edgecolor='white')
    if ap50_vals:
        v7_colors = ['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.6 else '#e74c3c'
                     for v in ap50_vals]
        ax9.bar(x + w/2, ap50_vals, w, label='v7', color=v7_colors, alpha=0.9, edgecolor='white')
        for i, (v6, v7) in enumerate(zip(V6_AP, ap50_vals)):
            ax9.text(x[i]+w/2, v7+0.01, f'{v7*100:.0f}%', ha='center', va='bottom', fontsize=8)
    ax9.set_xticks(x)
    ax9.set_xticklabels(CLASS_KO, rotation=35, fontsize=9)
    ax9.axhline(0.8, color='green', ls='--', alpha=0.5, lw=1.5)
    ax9.set_ylim(0, 1.15)
    ax9.legend(fontsize=9)
    make_ax(ax9, '⑨ v6 vs v7 클래스별 AP@0.5', xlabel='', ylabel='AP@0.5')

    ax10 = fig.add_subplot(4, 3, 10)
    ax10.axis('off')
    rows = []
    if 'map50'     in COL: rows.append(['최고 mAP@0.5',      f"{df[COL['map50']].max()*100:.2f}%"])
    if 'map50'     in COL: rows.append(['최고 mAP epoch',     str(int(epochs[df[COL['map50']].idxmax()]))])
    if 'map50_95'  in COL: rows.append(['최고 mAP@0.5:0.95',  f"{df[COL['map50_95']].max()*100:.2f}%"])
    if 'precision' in COL: rows.append(['최고 Precision',     f"{df[COL['precision']].max()*100:.2f}%"])
    if 'recall'    in COL: rows.append(['최고 Recall',        f"{df[COL['recall']].max()*100:.2f}%"])
    rows.append(['모델', 'YOLOv8x'])
    rows.append(['학습 전략', 'Phase1→Phase2'])
    rows.append(['총 Phase2 Epoch', str(len(df))])
    if ap50_vals:
        rows.append(['최고 클래스', f"{CLASS_KO[int(np.argmax(ap50_vals))]} ({max(ap50_vals)*100:.1f}%)"])
        rows.append(['최저 클래스', f"{CLASS_KO[int(np.argmin(ap50_vals))]} ({min(ap50_vals)*100:.1f}%)"])
        rows.append(['v6→v7 mAP 개선', f"{(np.mean(ap50_vals)-np.mean(V6_AP))*100:+.1f}%p"])
    tbl = ax10.table(cellText=rows, colLabels=['항목','값'],
                     cellLoc='center', loc='center', colWidths=[0.58,0.42])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.7)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#2C3E50')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#EBF5FB')
        cell.set_edgecolor('#BDC3C7')
    ax10.set_title('⑩ 학습 요약 (v7)', fontsize=11, fontweight='bold', pad=8)

    ax11 = fig.add_subplot(4, 3, 11)
    if 'lr' in COL:
        ax11.plot(epochs, df[COL['lr']], '#937860', lw=2)
        ax11.fill_between(epochs, df[COL['lr']], alpha=0.15, color='#937860')
    make_ax(ax11, '⑪ Learning Rate 스케줄', ylabel='LR')

    ax12 = fig.add_subplot(4, 3, 12)
    if all(k in COL for k in ['train_box','train_cls','val_box','val_cls']):
        train_total = df[COL['train_box']] + df[COL['train_cls']]
        val_total   = df[COL['val_box']]   + df[COL['val_cls']]
        ax12.plot(epochs, train_total, '#4C72B0', lw=2, label='Train 합계')
        ax12.plot(epochs, val_total,   '#C44E52', lw=2, ls='--', label='Val 합계')
        ax12.fill_between(epochs, train_total, val_total,
                          where=(val_total > train_total),
                          alpha=0.1, color='red', label='과적합 구간')
    make_ax(ax12, '⑫ 전체 Loss 합계 (과적합 진단)', ylabel='Loss 합계')
    ax12.legend(fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_path = SAVE_DIR / 'safewalk_v7_결과.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'\n그래프 저장: {save_path}')